# Assignment 10: Human Action Recognition with Video Swin Transformer

**Objective:** Learn to perform human action recognition from videos using the Video Swin Transformer neural network as a feature extractor.

**Instructions:**
1.  **Environment Setup:** This notebook requires a GPU runtime (T4 recommended).
2.  **Data:** Kinetics6-mini dataset.
3.  **Dependencies:** `decord`, `tensorflow`, `video_swin.py` (provided).


## 1. Setup Environment & Data

*   Install dependencies (`decord`).
*   Download pre-trained weights for Video Swin Transformer.
*   Extract the dataset.


## 2. Helper Functions & `video_swin.py`

Ensure `video_swin.py` is in the path. If running locally or with resource folder structure, add it to path.
Here we also define the `video_class.py` content as requested.


In [1]:
import sys
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import decord

# Add resources to system path to import video_swin
sys.path.append('resources') 

try:
    from video_swin import VideoSwinB
    print("VideoSwin imported successfully.")
except ImportError:
    print("Error: video_swin.py not found. Please ensure it is uploaded.")

# Global Config
RESOLUTION = 224
FRAME_COUNT = 32
BATCH_SIZE = 8
NUM_CLASSES = 6
LEARNING_RATE = 1e-4
EPOCHS = 5

# Map class names to indices
CLASS_NAMES = sorted([d for d in os.listdir('resources/kin6-mini/train') if os.path.isdir(os.path.join('resources/kin6-mini/train', d))])
CLASS_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}
print("Classes:", CLASS_MAP)


c:\Users\danie\Documents\Projects\ml4cv\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


VideoSwin imported successfully.
Classes: {'dining': 0, 'mowing_lawn': 1, 'pushing_car': 2}


## 3. `video_class.py` Content & Inspection

Below is the code typically found in `video_class.py` (reproduced here for study).

**Q2 Analysis:**
*   **prepare_dataset(path, frame_count, resolution, batch_size):**
    *   **Input Types:** `path` (str), `frame_count` (int), `resolution` (int), `batch_size` (int).
    *   **Purpose:**
        *   `path`: Directory containing class subfolders of videos.
        *   `frame_count`: Number of frames to extract per clip.
        *   `resolution`: Height/Width to resize frames to.
        *   `batch_size`: Number of videos per batch.
    *   **Nested Loops:**
        *   Outer `while True`: Infinite loop to allow the generator to run indefinitely (needed for `fit` with steps_per_epoch).
        *   Inner `for video_path in video_paths`: Iterates through shuffled list of all video files.
    *   **Yield:** Returns a tuple `(videos, labels)` representing one batch. This turns the function into a Python Generator, lazy-loading data.

**Q3 Analysis:**
*   **create_network(input_shape, num_classes):**
    *   It initializes the `VideoSwinB` backbone.
    *   Adds a **Classification Head**: `GlobalAveragePooling3D` -> `Dense(1024)` -> `Dropout` -> `Dense(num_classes)`.
    *   Compiles the model with Optimizer (Adam) and Loss (SparseCategoricalCrossentropy).


In [ ]:
def format_frames(frame, output_size):
    """
    Pad and resize an image from a video.
    Args:
      frame: Image that needs to resized and padded. 
      output_size: Pixel size of the output frame image.
    Return:
      Formatted frame with padding of size output_size.
    """
    frame = tf.image.convert_image_dtype(frame, tf.float32)
    frame = tf.image.resize_with_pad(frame, *output_size)
    return frame

def create_network(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    
    # Backbone
    backbone = VideoSwinB(include_rescaling=True, include_top=False, input_shape=input_shape)
    
    # Load weights (optional, if available matching structure)
    # backbone.load_weights('videoswin_base_kinetics400_classifier.weights.h5', by_name=True, skip_mismatch=True)
    
    x = backbone(inputs)
    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


## 4. Implement `get_clip` (Q4)

**Task:** Implement a function to read video, sample frames, handle short clips, and resize.

**Solution Logic:**
1.  Initialize `decord.VideoReader`.
2.  Get total frame count `len(vr)`.
3.  Calculate indices:
    *   If video is long enough, take uniform samples or random clip.
    *   If video is **too short** (< `frame_count`), use `np.linspace(0, total-1, frame_count)` to resample indices (repeating frames).
4.  Get frames using indices: `vr.get_batch(indices)`.
5.  Format frames using `format_frames`.


In [ ]:
def get_clip(file_path, frame_count, resolution):
    try:
        vr = decord.VideoReader(file_path)
        v_len = len(vr)
        
        if v_len >= frame_count:
            # Sample frames evenly or just take the first frame_count
            indices = np.linspace(0, v_len - 1, frame_count).astype(int)
        else:
            # Repeat frames if video is short
            indices = np.linspace(0, v_len - 1, frame_count).astype(int)
            
        frames = vr.get_batch(indices).asnumpy()
        formatted_frames = [format_frames(f, (resolution, resolution)) for f in frames]
        return np.stack(formatted_frames)
        
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        # Return zeros if fail
        return np.zeros((frame_count, resolution, resolution, 3))

def prepare_dataset(base_path, frame_count, resolution, batch_size):
    # Collect all video paths and labels
    video_paths = []
    labels = []
    
    for cls in CLASS_NAMES:
        cls_path = os.path.join(base_path, cls)
        files = [os.path.join(cls_path, f) for f in os.listdir(cls_path) if f.endswith('.mp4') or f.endswith('.avi')]
        video_paths.extend(files)
        labels.extend([CLASS_MAP[cls]] * len(files))
    
    video_paths = np.array(video_paths)
    labels = np.array(labels)
    num_samples = len(video_paths)
    
    while True:
        # Shuffle each epoch
        perm = np.random.permutation(num_samples)
        video_paths = video_paths[perm]
        labels = labels[perm]
        
        for i in range(0, num_samples, batch_size):
            batch_paths = video_paths[i : i + batch_size]
            batch_labels = labels[i : i + batch_size]
            
            batch_videos = []
            valid_labels = []
            
            for path, lbl in zip(batch_paths, batch_labels):
                clip = get_clip(path, frame_count, resolution)
                batch_videos.append(clip)
                valid_labels.append(lbl)
                
            yield np.array(batch_videos), np.array(valid_labels)


## 5. Training and Evaluation (Q5)

**Task:** Run classification on `kin6-mini` and report accuracy.


In [ ]:
# Setup Generators
train_path = 'resources/kin6-mini/train'
val_path = 'resources/kin6-mini/val'
test_path = 'resources/kin6-mini/test'

# Count samples to define steps_per_epoch
num_train = sum([len(files) for r, d, files in os.walk(train_path)])
num_val = sum([len(files) for r, d, files in os.walk(val_path)])
num_test = sum([len(files) for r, d, files in os.walk(test_path)])

print(f"Samples: Train={num_train}, Val={num_val}, Test={num_test}")

train_gen = prepare_dataset(train_path, FRAME_COUNT, RESOLUTION, BATCH_SIZE)
val_gen = prepare_dataset(val_path, FRAME_COUNT, RESOLUTION, BATCH_SIZE)
test_gen = prepare_dataset(test_path, FRAME_COUNT, RESOLUTION, BATCH_SIZE)

# Build Model
model = create_network((FRAME_COUNT, RESOLUTION, RESOLUTION, 3), NUM_CLASSES)
model.summary()

# Train
history = model.fit(
    train_gen,
    validation_data=val_gen,
    steps_per_epoch=num_train // BATCH_SIZE,
    validation_steps=num_val // BATCH_SIZE,
    epochs=EPOCHS
)

# Evaluate
print("\nTesting Model...")
loss, acc = model.evaluate(test_gen, steps=num_test // BATCH_SIZE)
print(f"Test Accuracy: {acc*100:.2f}%")
